# SQLDrift GRPO Training (Colab T4 — Qwen3-1.7B + QLoRA)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishesh-rathi/sql-drift-env/blob/main/sql_drift_grpo_training.ipynb)

A self-contained, **TRL-only** training notebook for SQLDrift. Modeled after Hugging Face TRL's official references:

- [`grpo_trl_lora_qlora.ipynb`](https://github.com/huggingface/trl/blob/main/examples/notebooks/grpo_trl_lora_qlora.ipynb) — GRPO + QLoRA on a free Colab T4
- [TRL OpenEnv integration](https://huggingface.co/docs/trl/openenv) — multi-turn tool-calling RL via `environment_factory`

**Stack** (no Unsloth, no HF Jobs):

| Component    | Choice                                                                                                                                                          |
| ------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Model        | `Qwen/Qwen3-1.7B` — Qwen3 dense base, Apache-2.0; smaller VRAM than 4B instruct variants; override with `SQL_DRIFT_MODEL_NAME` if you prefer another checkpoint |
| Trainer      | `trl>=1.2.0` `GRPOTrainer` with `environment_factory=SqlDriftToolEnv`                                                                                           |
| Quantization | `transformers.BitsAndBytesConfig` 4-bit nf4 (QLoRA)                                                                                                             |
| Adapter      | `peft.LoraConfig` r=16, α=32, target = 7 attention/MLP projections                                                                                              |
| Hardware     | Free Colab **T4 (16 GB)** — fp16, per_device=1, num_generations=2                                                                                               |

**What this notebook does:**

1. Installs `trl[peft] + bitsandbytes + jmespath` and clones the SQLDrift repo so we can import `SqlDriftToolEnv` and the curriculum builder.
2. Health-checks the deployed SQLDrift OpenEnv Space (set `SQL_DRIFT_ENV_URL`).
3. Builds a curriculum dataset of `(scenario_id, seed, budget_steps)` rows that TRL forwards to `SqlDriftToolEnv.reset(**kwargs)` per episode.
4. Loads Qwen3-1.7B with QLoRA 4-bit nf4, attaches LoRA via `peft_config`, and trains with multi-turn tool-using rollouts.
5. Writes durable evidence: `training/evidence/grpo_metrics.csv`, `grpo_loss_curve.png`, `grpo_reward_curve.png`.

**Before running:** in Colab, switch the runtime to a **T4 GPU** (Runtime → Change runtime type → T4 GPU), then set `SQL_DRIFT_ENV_URL` and (if cloning) `SQL_DRIFT_REPO_URL` in the cells below.


## 1. Install training stack & clone the SQLDrift repo

We install **TRL with the PEFT extra** (pulls `trl`, `transformers`, `peft`, `accelerate`, `datasets`, `tokenizers`, `torch`), plus `bitsandbytes` for 4-bit nf4 quantization and `jmespath` (TRL needs it to parse tool-call responses when `environment_factory` is set).

If the notebook is launched fresh in Colab (so this directory has no `pyproject.toml`), it clones the SQLDrift repo and `cd`s into it; otherwise it stays put. Set `SQL_DRIFT_REPO_URL` if you forked the repo.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# 1a. Install the TRL training stack (GRPOTrainer + PEFT extras + bitsandbytes).
#     `trl[peft]` pulls trl, transformers, peft, accelerate, datasets, tokenizers, torch.
#     `jmespath` is required by TRL's GRPOTrainer when environment_factory / tools are set.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-Uq",
        "trl[peft]>=1.2.0",
        "git+https://github.com/huggingface/transformers.git@main",
        "bitsandbytes>=0.46.1",
        "jmespath>=1.0",
        "tensorboard>=2.20",
        "matplotlib",
        "pandas",
    ]
)

# 1b. Clone the SQLDrift repo if we're not already inside it. Defaults to the public
#     repo URL but respects SQL_DRIFT_REPO_URL if you forked it.
REPO_URL = os.environ.get(
    "SQL_DRIFT_REPO_URL", "https://github.com/vishesh-rathi/sql-drift-env"
).strip()
repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    subprocess.check_call(["git", "clone", REPO_URL, "sql_drift_env"])
    os.chdir("sql_drift_env")
    repo_root = Path.cwd()

# 1c. Install the SQLDrift base runtime deps (duckdb, sqlglot, pydantic, openenv-core,
#     openai, python-dotenv, huggingface-hub) by pip-installing the package without its
#     [train] extra — we already installed the trainer stack above.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-Uq", "-e", "."]
)

# 1d. Make the repo's flat-layout modules importable: `from training.tool_env import ...`,
#     `from client import SqlDriftEnv`, etc. The repo uses sibling-package imports.
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Avoid tokenizer worker deadlocks when Colab/fork interact with DataLoader.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print(f"Repo root      : {repo_root}")
print(f"Python version : {sys.version.split()[0]}")
import torch  # noqa: E402

print(
    f"Torch / CUDA   : torch={torch.__version__} "
    f"cuda_available={torch.cuda.is_available()} "
    f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
)
assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, then re-run."
)

Repo root      : /content/sql_drift_env
Python version : 3.12.13
Torch / CUDA   : torch=2.10.0+cu128 cuda_available=True device=Tesla T4


## 2. Point at the deployed SQLDrift OpenEnv Space

`SqlDriftToolEnv` opens a WebSocket session to a deployed SQLDrift Space for every rollout. Set `SQL_DRIFT_ENV_URL` to your Space URL (`https://<user>-sql-drift-env.hf.space`) and we'll health-check it before training starts.

> **Concurrency tip** — TRL opens N concurrent sessions (one per `num_generations`). Per the [TRL OpenEnv docs](https://huggingface.co/docs/trl/openenv#server-concurrency), duplicate the Space to your own account if the original doesn't allow concurrent sessions.


In [ ]:
import urllib.request

# Default to the Space already deployed for this project; override via env var.
ENV_BASE_URL = os.environ.get(
    "SQL_DRIFT_ENV_URL", "https://visheshrathi-sql-drift-env.hf.space"
).strip()
os.environ["SQL_DRIFT_ENV_URL"] = ENV_BASE_URL  # SqlDriftToolEnv reads this internally.

health_url = ENV_BASE_URL.rstrip("/") + "/health"
with urllib.request.urlopen(health_url, timeout=30) as response:
    body = response.read().decode("utf-8")

print(f"OpenEnv URL    : {ENV_BASE_URL}")
print(f"Health check OK: {health_url}")
print(body[:500])

OpenEnv URL    : https://visheshrathi-sql-drift-env.hf.space
Health check OK: https://visheshrathi-sql-drift-env.hf.space/health
{"status":"healthy"}


## 3. Build the curriculum dataset

`training.grpo_train.build_dataset` produces a `datasets.Dataset` whose rows carry `prompt` plus `(scenario_id, seed, budget_steps, enable_dba_oracle)`. Per the [TRL OpenEnv docs](https://huggingface.co/docs/trl/openenv#environment-class-requirements), TRL forwards every dataset column to `SqlDriftToolEnv.reset(**kwargs)` so each rollout pins to a reproducible curriculum slot.

Defaults are sized for a Colab T4 + free-tier OpenEnv Space; override the env vars to scale up.


In [ ]:
import json
import os
from dataclasses import asdict
from pathlib import Path

from training.config import ALL_SCENARIOS, CurriculumConfig, GRPOConfig
from training.grpo_train import build_dataset

# Colab edge case: section §1 should have cd'd into the clone, but if you
# re-run from a stale cwd, recover so `training.*` imports work.
if not Path("training/grpo_train.py").exists() and Path("/content/sql_drift_env/training/grpo_train.py").exists():
    os.chdir("/content/sql_drift_env")

MODEL_NAME = os.environ.get("SQL_DRIFT_MODEL_NAME", "Qwen/Qwen3-1.7B")
MAX_STEPS = int(os.environ.get("SQL_DRIFT_GRPO_MAX_STEPS", "80"))
GROUP_SIZE = int(os.environ.get("SQL_DRIFT_GRPO_GROUP_SIZE", "4"))
GRAD_ACCUM = int(os.environ.get("SQL_DRIFT_GRAD_ACCUM", "4"))
SEED = int(os.environ.get("SQL_DRIFT_GRPO_SEED", "7"))
LEARNING_RATE = float(os.environ.get("SQL_DRIFT_LR", "5e-6"))

OUTPUT_DIR = "outputs/checkpoints/grpo_qwen3_17b"
EVIDENCE_DIR = Path("training/evidence")
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

cfg = GRPOConfig(
    model_name=MODEL_NAME,
    env_base_url=ENV_BASE_URL,
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    group_size=GROUP_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=min(10, max(1, MAX_STEPS // 10)),
    save_steps=max(20, MAX_STEPS // 4),
    logging_steps=1,
    seed=SEED,
    max_seq_length=4096,
    # Total token budget for the whole multi-turn rollout (NOT per turn). 1024 is
    # too tight for long tool loops and can make a single GRPO step look "stuck".
    max_completion_length=int(os.environ.get("SQL_DRIFT_MAX_COMPLETION_LEN", "2048")),
    max_tool_calling_iterations=int(os.environ.get("SQL_DRIFT_MAX_TOOL_ITERS", "32")),
    # Qwen3-1.7B sampler defaults (tune if you swap checkpoints).
    temperature=1,
    top_p=0.8,
    fp16=True,
    bf16=False,
    load_in_4bit=True,
    lora_r=16,
    lora_alpha=32,
    curriculum=CurriculumConfig(
        scenarios=ALL_SCENARIOS,
        mode="weighted",
        # Weight the trickier drift scenarios (last four) more heavily.
        weights=(1, 1, 1, 1, 1, 1, 2, 2, 2, 2),
    ),
)

# TRL's `environment_factory` instantiates one env per generation, so the
# dataset just needs `max_steps * group_size` rows for one full epoch.
NUM_ROWS = max(cfg.max_steps * cfg.group_size, cfg.group_size)
train_dataset = build_dataset(cfg, num_rows=NUM_ROWS, seed=cfg.seed)

(EVIDENCE_DIR / "grpo_config.json").write_text(json.dumps(asdict(cfg), indent=2))
print(f"Model        : {cfg.model_name}")
print(f"Max steps    : {cfg.max_steps}  (group_size={cfg.group_size}, grad_accum={cfg.gradient_accumulation_steps})")
print(f"Dataset rows : {len(train_dataset)}  (columns={list(train_dataset.column_names)})")
print(f"First prompt : {train_dataset[0]['prompt'][0]['content'][:160]}…")

Model        : Qwen/Qwen3-1.7B
Max steps    : 80  (group_size=4, grad_accum=4)
Dataset rows : 320  (columns=['prompt', 'scenario_id', 'seed', 'budget_steps', 'enable_dba_oracle'])
First prompt : You are a senior SQL engineer operating an analytical database that is under live schema and business-rule drift. Your job is to repair and optimize a slow base…


## 4. Load Qwen3-1.7B with QLoRA (4-bit nf4)

Identical recipe to Hugging Face TRL's [`grpo_trl_lora_qlora.ipynb`](https://github.com/huggingface/trl/blob/main/examples/notebooks/grpo_trl_lora_qlora.ipynb): plain `AutoModelForCausalLM.from_pretrained(...)` with a `BitsAndBytesConfig` quantization spec, then a `LoraConfig` adapter passed to `GRPOTrainer(peft_config=...)`.

Memory budget on a Colab T4 (16 GB): 1.7B in 4-bit nf4 is lighter than 4B—roughly **~1.5–2 GB weights** plus activations and KV cache during multi-turn rollouts—so you retain comfortable headroom at `num_generations=2` and can sometimes raise batch-related knobs if the Space keeps up.


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4a. Quantization config — 4-bit nf4 with double quantization (QLoRA recipe).
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # T4 supports fp16, not bf16.
)

# 4b. Load the base model. `dtype="float32"` matches the TRL reference notebook
#     (the inner compute is fp16 via bnb_4bit_compute_dtype above; only the
#     untouched-by-quantization layers sit in fp32 for stability).
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    attn_implementation="sdpa",
    dtype=torch.float16,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

# Qwen3 checkpoints use a chat template path that TRL wires via `qwen3_schema` for
# tool-call parsing (`add_response_schema`). Keep this aligned with TRL when you
# upgrade. Source:
# https://github.com/huggingface/trl/blob/main/trl/chat_template_utils.py
from trl.chat_template_utils import qwen3_schema
tokenizer.response_schema = qwen3_schema

# 4c. PEFT LoRA config — applied by GRPOTrainer when we pass `peft_config=...`.
peft_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=list(cfg.lora_target_modules),
    bias="none",
    task_type="CAUSAL_LM",
)

model.enable_input_require_grads()

# Sanity-check VRAM before training so we can compare with peak after.
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory = round(gpu_stats.total_memory / 1024**3, 3)
print(f"GPU            : {gpu_stats.name}  (max {max_memory} GB)")
print(f"Reserved (pre) : {start_gpu_memory} GB")
print(f"PEFT LoRA      : r={cfg.lora_r} alpha={cfg.lora_alpha} target={list(cfg.lora_target_modules)}")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

GPU            : Tesla T4  (max 14.563 GB)
Reserved (pre) : 11.857 GB
PEFT LoRA      : r=16 alpha=32 target=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


In [ ]:
# Optional: only needed if you re-ran the notebook from a different cwd than §1
# (the install cell already `cd`s into the clone and fixes `sys.path`).
%cd /content/sql_drift_env
import sys
if "." not in sys.path:
    sys.path.insert(0, ".")

/content


## 5. Configure GRPO + reward function

We use the canonical TRL `environment_factory` API ([docs](https://huggingface.co/docs/trl/openenv)). `SqlDriftToolEnv` (already in this repo) is the env class:

- `__init__(self, *, env_url=None)` — opens a sync WebSocket session against the deployed Space.
- `reset(self, **kwargs)` — TRL forwards every dataset column as a kwarg; we pin scenario/seed/budget here.
- Public methods (`list_tables`, `describe_table`, `sample_rows`, `run_query`, `explain_query`, `read_changelog`, `submit_rewrite`, `consult_dba`) are auto-discovered as tools.
- `self.episode_return` accumulates per-step reward across the rollout.

The reward function reads `env.episode_return` from each environment instance — exactly the [pattern](https://huggingface.co/docs/trl/openenv#reward-functions) the TRL OpenEnv docs specify.


In [ ]:
from functools import partial
from pathlib import Path

from trl import GRPOConfig as TRLGRPOConfig
from trl import GRPOTrainer

from training.grpo_train import reward_from_environments
from training.tool_env import SqlDriftToolEnv

# Per the TRL OpenEnv guide, `max_completion_length` is the TOTAL multi-turn
# token budget (assistant + tool results combined), not a per-turn cap.
# `max_tool_calling_iterations` MUST be set: TRL's default is unbounded and a
# chatty model can keep requesting tools for hours per optimizer step.
training_args = TRLGRPOConfig(
    output_dir=cfg.output_dir,
    learning_rate=cfg.learning_rate,
    max_steps=cfg.max_steps,
    per_device_train_batch_size=1,            # T4-safe; TRL needs batch * grad_accum % num_generations == 0
    num_generations=cfg.group_size,           # 2 is the sweet spot for T4 + 4-bit
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    warmup_steps=cfg.warmup_steps,
    max_completion_length=cfg.max_completion_length,
    max_tool_calling_iterations=cfg.max_tool_calling_iterations,
    temperature=cfg.temperature,
    top_p=cfg.top_p,
    fp16=cfg.fp16,
    bf16=cfg.bf16,
    optim="paged_adamw_8bit",                 # Pairs with QLoRA — keeps optimizer state off the hot path.
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    log_completions=True,
    report_to=["tensorboard"],
    logging_dir=str(Path(cfg.output_dir) / "tb"),
    seed=cfg.seed,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    chat_template_kwargs={"enable_thinking": False},
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=reward_from_environments,
    environment_factory=partial(SqlDriftToolEnv, env_url=ENV_BASE_URL),
    peft_config=peft_config,
)

print(f"Trainer ready. Tools auto-discovered on SqlDriftToolEnv:")
print("  list_tables, describe_table, sample_rows, run_query, explain_query,")
print("  read_changelog, submit_rewrite, consult_dba")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Trainer ready. Tools auto-discovered on SqlDriftToolEnv:
  list_tables, describe_table, sample_rows, run_query, explain_query,
  read_changelog, submit_rewrite, consult_dba


/tmp/ipykernel_504/714143961.py:46: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(


## 6. Train

`trainer.train()` runs the full GRPO loop: for each step, TRL spins up `num_generations` `SqlDriftToolEnv` instances in parallel, generates multi-turn rollouts (model decides which tools to call, TRL parses + executes them, results feed back), reads `episode_return` from each instance, computes group-relative advantages, and updates the LoRA adapter.

> Expect **~10–15 min/step** on a free T4 with QLoRA at `num_generations=2` and `max_completion_length=2048` (rollouts dominate the wall clock — every tool call hops the WS to the Space). 80 steps ≈ 12–20 hours, well within Colab's 12-hour session window if you split across runs (TRL checkpoints LoRA adapters every `save_steps`).


In [ ]:
print(
    "Starting trainer.train() — the first step can take *several* minutes: "
    "TRL runs full multi-turn tool rollouts to your Space and then backprop. "
    "Watch for `loss` in the logs; it is not hung if GPU util cycles.",
    flush=True,
)
trainer_stats = trainer.train()

# Save the LoRA adapter so a re-run / inference loop can attach it back.
trainer.save_model(cfg.output_dir)

history = list(getattr(trainer.state, "log_history", []))
print(f"Captured {len(history)} trainer log records.")
print(f"Adapter dir : {cfg.output_dir}")

# VRAM peak post-training, for comparison with the pre-train reading.
used_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
used_for_lora = round(used_memory - start_gpu_memory, 3)
used_pct = round(used_memory / max_memory * 100, 3)
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"VRAM peak    : {used_memory} GB ({used_pct}% of {max_memory} GB)")
print(f"VRAM Δ train : {used_for_lora} GB")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
1,0.000000
2,0.175236


╭──────────────────────────────────────────────────── Step 1 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                           ┃ Completion                        ┃ reward_from_environments ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                           │ <tool_call>                       │                     0.03 │      0.00 │ │
│ │ You are a senior SQL engineer    │ {"name": "list_tables",           │                          │           │ │
│ │ operating an analytical database │ "arguments": {"table": ""}}       │                          │           │ │
│ │ that is under live schema and    │ </tool_call>                      │                          │           │ │
│ │ business-rule drift. Your job is │ user                              │                          │           │ │
│ │ to repair and optimize a slow    │ <tool_response>                   │                          │           │ │
│ │ baseline SELECT under tight step │ {'error':                         │                          │           │ │
│ │ and runtime budgets. Prefer      │ "SqlDriftToolEnv.list_tables()    │                          │           │ │
│ │ read-only tools; never emit DDL  │ got an unexpected keyword         │                          │           │ │
│ │ or DML (INSERT/UPDATE/DELETE).   │ argument 'table'"}                │                          │           │ │
│ │ When a changelog is published,   │ </tool_response>                  │                          │           │ │
│ │ treat it as authoritative.       │ assistant                         │                          │           │ │
│ │                                  │ <think>                           │                          │           │ │
│ │ Tools available (exact JSON      │                                   │                          │           │ │
│ │ shapes enforced by the env):     │ </think>                          │                          │           │ │
│ │ - list_tables(): Enumerate       │                                   │                          │           │ │
│ │ tables visible to the session    │ <tool_call>                       │                          │           │ │
│ │ (cheap, always safe).            │ {"name": "consult_dba",           │                          │           │ │
│ │ - describe_table(table: str):    │ "arguments": {"question": "Can    │                          │           │ │
│ │ Return column names + types for  │ you help me diagnose the schema   │                          │           │ │
│ │ one table.                       │ and sample data for the events    │                          │           │ │
│ │ - sample_rows(table: str, limit: │ table?"}}                         │                          │           │ │
│ │ int ∈ [1, 5] = 5): Peek at up to │ </tool_call>                      │                          │           │ │
│ │ 5 rows for fast schema           │ user                              │                          │           │ │
│ │ intuition.                       │ <tool_response>                   │                          │           │ │
│ │ - run_query(sql: str): Execute a │ error[invalid_tool_argument]:     │                          │           │ │
│ │ read-only SELECT against the     │ consult_dba disabled — set        │                          │           │ │
│ │ live database. Timing counts     │ enable_dba_oracle=True at reset() │                          │           │ │
│ │ toward the step budget;          │ </tool_response>                  │                          │           │ │
│ │ repeat-failing queries are       │ assistant                         │                          │           │ │
│ │ penalised.                       │ <think>          

╭──────────────────────────────────────────────────── Step 2 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                           ┃ Completion                        ┃ reward_from_environments ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                           │ <tool_call>                       │                     0.08 │      0.86 │ │
│ │ You are a senior SQL engineer    │ {"name": "list_tables",           │                          │           │ │
│ │ operating an analytical database │ "arguments": {"table": ""}}       │                          │           │ │
│ │ that is under live schema and    │ </tool_call>                      │                          │           │ │
│ │ business-rule drift. Your job is │ user                              │                          │           │ │
│ │ to repair and optimize a slow    │ <tool_response>                   │                          │           │ │
│ │ baseline SELECT under tight step │ {'error':                         │                          │           │ │
│ │ and runtime budgets. Prefer      │ "SqlDriftToolEnv.list_tables()    │                          │           │ │
│ │ read-only tools; never emit DDL  │ got an unexpected keyword         │                          │           │ │
│ │ or DML (INSERT/UPDATE/DELETE).   │ argument 'table'"}                │                          │           │ │
│ │ When a changelog is published,   │ </tool_response>                  │                          │           │ │
│ │ treat it as authoritative.       │ assistant                         │                          │           │ │
│ │                                  │ <think>                           │                          │           │ │
│ │ Tools available (exact JSON      │                                   │                          │           │ │
│ │ shapes enforced by the env):     │ </think>                          │                          │           │ │
│ │ - list_tables(): Enumerate       │                                   │                          │           │ │
│ │ tables visible to the session    │ <tool_call>                       │                          │           │ │
│ │ (cheap, always safe).            │ {"name": "list_tables",           │                          │           │ │
│ │ - describe_table(table: str):    │ "arguments": {}}                  │                          │           │ │
│ │ Return column names + types for  │ </tool_call>                      │                          │           │ │
│ │ one table.                       │ user                              │                          │           │ │
│ │ - sample_rows(table: str, limit: │ <tool_response>                   │                          │           │ │
│ │ int ∈ [1, 5] = 5): Peek at up to │ posts                             │                          │           │ │
│ │ 5 rows for fast schema           │ </tool_response>                  │                          │           │ │
│ │ intuition.                       │ assistant                         │                          │           │ │
│ │ - run_query(sql: str): Execute a │ <think>                           │                          │           │ │
│ │ read-only SELECT against the     │                                   │                          │           │ │
│ │ live database. Timing counts     │ </think>                          │                          │           │ │
│ │ toward the step budget;          │                                   │                          │           │ │
│ │ repeat-failing queries are       │ <tool_call>                       │                          │           │ │
│ │ penalised.                       │ {"name": "describ

╭──────────────────────────────────────────────────── Step 3 ─────────────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┓ │
│ ┃ Prompt                           ┃ Completion                        ┃ reward_from_environments ┃ Advantage ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━┩ │
│ │ system                           │ <tool_call>                       │                    -0.30 │      0.00 │ │
│ │ You are a senior SQL engineer    │ {"name": "list_tables",           │                          │           │ │
│ │ operating an analytical database │ "arguments": {"table": ""}}       │                          │           │ │
│ │ that is under live schema and    │ </tool_call>                      │                          │           │ │
│ │ business-rule drift. Your job is │ user                              │                          │           │ │
│ │ to repair and optimize a slow    │ <tool_response>                   │                          │           │ │
│ │ baseline SELECT under tight step │ {'error':                         │                          │           │ │
│ │ and runtime budgets. Prefer      │ "SqlDriftToolEnv.list_tables()    │                          │           │ │
│ │ read-only tools; never emit DDL  │ got an unexpected keyword         │                          │           │ │
│ │ or DML (INSERT/UPDATE/DELETE).   │ argument 'table'"}                │                          │           │ │
│ │ When a changelog is published,   │ </tool_response>                  │                          │           │ │
│ │ treat it as authoritative.       │ assistant                         │                          │           │ │
│ │                                  │ <think>                           │                          │           │ │
│ │ Tools available (exact JSON      │                                   │                          │           │ │
│ │ shapes enforced by the env):     │ </think>                          │                          │           │ │
│ │ - list_tables(): Enumerate       │                                   │                          │           │ │
│ │ tables visible to the session    │ <tool_call>                       │                          │           │ │
│ │ (cheap, always safe).            │ {"name": "consult_dba",           │                          │           │ │
│ │ - describe_table(table: str):    │ "arguments": {"question": "Can    │                          │           │ │
│ │ Return column names + types for  │ you help me diagnose the schema   │                          │           │ │
│ │ one table.                       │ and query performance for the     │                          │           │ │
│ │ - sample_rows(table: str, limit: │ pageviews table?"}}               │                          │           │ │
│ │ int ∈ [1, 5] = 5): Peek at up to │ </tool_call>                      │                          │           │ │
│ │ 5 rows for fast schema           │ user                              │                          │           │ │
│ │ intuition.                       │ <tool_response>                   │                          │           │ │
│ │ - run_query(sql: str): Execute a │ error[invalid_tool_argument]:     │                          │           │ │
│ │ read-only SELECT against the     │ consult_dba disabled — set        │                          │           │ │
│ │ live database. Timing counts     │ enable_dba_oracle=True at reset() │                          │           │ │
│ │ toward the step budget;          │ </tool_response>                  │                          │           │ │
│ │ repeat-failing queries are       │ assistant                         │                          │           │ │
│ │ penalised.                       │ <think>          

## 7. Save evidence (CSV + loss/reward PNG curves)

Evaluators need committed image files, not just inline notebook plots. This cell writes the trainer's log history to `training/evidence/grpo_metrics.csv` and renders the loss + reward curves to PNG.

After this cell finishes:

```bash
git add sql_drift_grpo_training.ipynb \
        training/evidence/grpo_metrics.csv \
        training/evidence/grpo_loss_curve.png \
        training/evidence/grpo_reward_curve.png \
        training/evidence/grpo_config.json
git commit -m "Add SQLDrift GRPO Qwen3-1.7B evidence"
```


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(history)
if df.empty:
    raise RuntimeError("Trainer log history is empty; training did not emit metrics.")

if "step" not in df.columns:
    df["step"] = range(len(df))

metrics_csv = EVIDENCE_DIR / "grpo_metrics.csv"
df.to_csv(metrics_csv, index=False)


def _numeric_columns(frame: pd.DataFrame) -> dict[str, pd.Series]:
    out: dict[str, pd.Series] = {}
    for column in frame.columns:
        series = pd.to_numeric(frame[column], errors="coerce")
        if series.notna().any():
            out[column] = series
    return out


def _choose_metric(
    frame: pd.DataFrame,
    *,
    preferred: list[str],
    required_token: str,
    exclude_tokens: tuple[str, ...] = (),
) -> tuple[str, pd.Series]:
    numeric = _numeric_columns(frame)
    for column in preferred:
        if column in numeric:
            return column, numeric[column]
    for column, series in numeric.items():
        lowered = column.lower()
        if required_token in lowered and not any(token in lowered for token in exclude_tokens):
            return column, series
    available = ", ".join(sorted(numeric))
    raise RuntimeError(
        f"Could not find a numeric {required_token!r} metric. Available: {available}"
    )


def _plot_curve(column: str, values: pd.Series, ylabel: str, title: str, path: Path) -> None:
    plot_df = pd.DataFrame(
        {"step": pd.to_numeric(df["step"], errors="coerce"), "value": values}
    ).dropna()
    if plot_df.empty:
        raise RuntimeError(f"Metric {column!r} has no plottable values.")

    plt.figure(figsize=(8, 4.5))
    plt.plot(plot_df["step"], plot_df["value"], marker="o", linewidth=1.4, label=column)
    if len(plot_df) >= 5:
        window = min(10, max(2, len(plot_df) // 5))
        rolling = plot_df["value"].rolling(window=window, min_periods=1).mean()
        plt.plot(plot_df["step"], rolling, linewidth=2.2, label=f"{window}-point rolling mean")
    plt.xlabel("GRPO step")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()


loss_col, loss_values = _choose_metric(
    df, preferred=["loss", "train_loss"], required_token="loss"
)
reward_col, reward_values = _choose_metric(
    df,
    preferred=["reward", "rewards/mean", "mean_reward", "reward_mean", "train/reward"],
    required_token="reward",
    exclude_tokens=("std", "variance"),
)

loss_png = EVIDENCE_DIR / "grpo_loss_curve.png"
reward_png = EVIDENCE_DIR / "grpo_reward_curve.png"
_plot_curve(loss_col, loss_values, "loss", "SQLDrift GRPO Loss Curve", loss_png)
_plot_curve(reward_col, reward_values, "episode reward", "SQLDrift GRPO Reward Curve", reward_png)

print(f"Wrote metrics    : {metrics_csv}")
print(f"Wrote loss curve : {loss_png}")
print(f"Wrote reward     : {reward_png}")